In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import re 

# Load PPI index

In [2]:
df_ppi_index = pd.read_csv("/Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/tables/Index data/PPI.csv", 
sep=";")

In [3]:
df_ppi_index.head()

,Country,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Albania,..,..,..,..,..,..,..,"101,1",98,"98,2","98,9","100,1",100,"97,9","99,9","117,3","126,7","128,1",..
1,Andorra,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..
2,Armenia,"47,2","48,8","52,3",64,"69,9","74,8","78,3","84,9","84,3","85,6","87,4","87,5",88,91,100,"102,6","104,2","107,7","111,8"
3,Austria,"84,3","88,4","86,8","90,3","94,6","95,5","94,5","93,1","91,1","88,9","90,6","93,3","93,4","91,5",100,"126,5","129,9","121,2","118,8"
4,Azerbaijan,"38,1","43,8","38,3","48,6","53,1","54,4","52,2","49,8","34,5","44,1","60,4",76,"78,4",59,100,"184,4","164,2","164,2","148,4"


## Transform to long format instead of wide

In [4]:
df_ppi_index_long = df_ppi_index.melt(
id_vars=['Country'], 
    var_name='Year', 
    value_name='PPI_Value'
)

In [5]:
df_ppi_index_long.head()

,Country,Year,PPI_Value
0,Albania,2007,..
1,Andorra,2007,..
2,Armenia,2007,"47,2"
3,Austria,2007,"84,3"
4,Azerbaijan,2007,"38,1"


## Convert to correct format

In [6]:
df_ppi_index_long['PPI_Value'] = (
    df_ppi_index_long['PPI_Value']
    .replace('..', np.nan)          
    .str.replace(',', '.')          
)

In [7]:
df_ppi_index_long['Year'] = pd.to_numeric(df_ppi_index_long['Year'])
df_ppi_index_long['PPI_Value'] = pd.to_numeric(df_ppi_index_long['PPI_Value'])

In [8]:
print(df_ppi_index_long.dtypes)
df_ppi_index_long.head()

Country          str
Year           int64
PPI_Value    float64
dtype: object


,Country,Year,PPI_Value
0,Albania,2007,NaN
1,Andorra,2007,NaN
2,Armenia,2007,47.2
3,Austria,2007,84.3
4,Azerbaijan,2007,38.1


# LPI Index

In [9]:

df_lpi= pd.read_csv("/Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/tables/Index data/LPI_Historical_Aligned.csv")

In [10]:
df_lpi.head()

,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,Year
0,Singapore,SGP,4.190336,3.89744,4.26957,4.03540,4.21495,4.24771,4.52941,2007
1,Netherlands,NLD,4.177695,3.99225,4.29032,4.04878,4.25000,4.13636,4.38235,2007
2,Germany,DEU,4.098695,3.88279,4.19133,3.90984,4.20728,4.11864,4.32727,2007
3,Sweden,SWE,4.075383,3.84615,4.11321,3.89796,4.06122,4.15217,4.43182,2007
4,Austria,AUT,4.062574,3.83333,4.06061,3.96667,4.13333,3.96552,4.44444,2007


In [11]:
df_lpi['Country'] = df_lpi['Country'].str.strip()
df_ppi_index_long['Country'] = df_ppi_index_long['Country'].str.strip()

In [12]:
df_macro_index = pd.merge(
    df_lpi, 
    df_ppi_index_long, 
    on=['Country', 'Year'], 
    how='outer' 
)

In [13]:
df_macro_index.head(20)

,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,Year,PPI_Value
0,Afghanistan,AFG,1.211669,1.300000,1.100000,1.222220,1.250000,1.000000,1.375000,2007,NaN
1,Afghanistan,AFG,2.243160,2.223085,1.873596,2.244916,2.090449,2.365503,2.605503,2010,NaN
2,Afghanistan,AFG,2.297272,2.333617,2.003611,2.332182,2.162592,2.095155,2.795259,2012,NaN
3,Afghanistan,AFG,2.069573,2.163453,1.818951,1.986686,2.119709,1.847776,2.482138,2014,NaN
4,Afghanistan,AFG,1.948565,1.734900,1.807143,2.104431,1.919246,1.697024,2.382438,2018,NaN
5,Afghanistan,AFG,1.900000,2.100000,1.700000,1.800000,2.000000,1.600000,2.300000,2023,NaN
6,Albania,ALB,2.081376,2.000000,2.333330,2.333330,2.000000,1.666670,2.125000,2007,NaN
7,Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008,NaN
8,Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009,NaN
9,Albania,ALB,2.459047,2.071429,2.144558,2.644558,2.394558,2.394558,3.012585,2010,NaN


We assume the 2012 logistics environment persists until the 2014 report.

In [14]:
df_macro_index = df_macro_index.sort_values(['Country', 'Year'])
columns_to_fill = ['LPI_Score', 'Customs', 'Infrastructure', 'International_Shipments', 
                   'Logistics_Competence', 'Tracking_Tracing', 'Timeliness']

Fill gaps within each country group so 2013 gets 2012's data

In [15]:
df_macro_index[columns_to_fill] = df_macro_index .groupby('Country')[columns_to_fill].ffill()

Filter such that we only keep years we actually needs

In [16]:
df_macro_index = df_macro_index.dropna(subset=['LPI_Score', 'PPI_Value'], how='all')

In [17]:
df_macro_index.head(20)

,Country,Code,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,Year,PPI_Value
0,Afghanistan,AFG,1.211669,1.300000,1.100000,1.222220,1.250000,1.000000,1.375000,2007,NaN
1,Afghanistan,AFG,2.243160,2.223085,1.873596,2.244916,2.090449,2.365503,2.605503,2010,NaN
2,Afghanistan,AFG,2.297272,2.333617,2.003611,2.332182,2.162592,2.095155,2.795259,2012,NaN
3,Afghanistan,AFG,2.069573,2.163453,1.818951,1.986686,2.119709,1.847776,2.482138,2014,NaN
4,Afghanistan,AFG,1.948565,1.734900,1.807143,2.104431,1.919246,1.697024,2.382438,2018,NaN
5,Afghanistan,AFG,1.900000,2.100000,1.700000,1.800000,2.000000,1.600000,2.300000,2023,NaN
6,Albania,ALB,2.081376,2.000000,2.333330,2.333330,2.000000,1.666670,2.125000,2007,NaN
7,Albania,NaN,2.081376,2.000000,2.333330,2.333330,2.000000,1.666670,2.125000,2008,NaN
8,Albania,NaN,2.081376,2.000000,2.333330,2.333330,2.000000,1.666670,2.125000,2009,NaN
9,Albania,ALB,2.459047,2.071429,2.144558,2.644558,2.394558,2.394558,3.012585,2010,NaN


In [18]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

from master_thesis.config import DATA_RAW

df_macro_index.to_csv(DATA_RAW / "macro_indexs.csv", index=False)
print("Saved to:", DATA_RAW / "macro_indexs.csv")




Saved to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/macro_indexs.csv
